# Datlinger17 processing decisions

Executable reconstruction layer for task `t_60ced1f1`. The notebook distinguishes immutable source facts, source-to-Lamin parity, the accepted append-only OBS revision, and replay/readback evidence. It performs no network or registry writes.

In [ ]:
import json
from pathlib import Path

ROOT = Path('artifacts/schema_audit/real_dataset_curation_20260722/scperturb_datlinger17/t_60ced1f1')
source = json.loads((ROOT / 'source_manifest.json').read_text())
plan = json.loads((ROOT / 'plan_receipt.json').read_text())
live = json.loads((ROOT / 'live_receipt.json').read_text())
verify = json.loads((ROOT / 'verify_receipt.json').read_text())
assert {item['sha256'] for item in source['files']} == {
    '3e0bb8554fdd6f732ec039e703f685631334e9c06029864e81594babc8def0af',
    '3acaf07ca5b5cb2fde9b957ae9e6f0b27a6df267b013ba9d818931b11ce54c44',
}
assert source['denominator_accounting']['source_and_accepted_observations'] == 5905
assert source['denominator_accounting']['guide_library_rows'] == 116
assert len(source['samples']) == 11


## Accepted source-backed semantics

All 5,905 accepted rows join in exact order to the GEO header after the upstream AnnData uniqueness rule. Every experimental and control guide ID maps to the 116-row primary-paper Supplementary Table 2. The primary paper supports CRISPR knockout semantics and a 4 h anti-CD3/CD28 versus continued-starvation condition. `raw_counts` is accepted only from all-value equality against the GEO digital-expression matrix.

In [ ]:
for receipt in (plan, live, verify):
    assert receipt['format'] == 'pert-gym.dataset-e2e-v3/v1'
    assert receipt['contract_sha256'] == 'bb786f6619c8a395575fe88c67f3c75bd4c6a545107c1535b02db88be02505a6'
    assert receipt['task_id'] == 't_60ced1f1'
    assert receipt['real_dataset_id'] == 'scperturb/datlinger17'
    assert receipt['status'] == 'PASS'
    after = receipt['member_after']
    assert after['source_join']['join_mismatch_count'] == 0
    assert after['source_join']['canonical_source_accession'] == 'GSE92872'
    assert after['source_join']['preserved_source_accession_rows'] == 5905
    assert after['source_join']['preserved_source_accession_values'] == ['datlinger17']
    assert after['source_join']['guide_sequence_known_rows'] == 5905
    assert after['x_source_parity']['source_value_mismatch_count'] == 0
    assert after['x_source_parity']['raw_counts'] is True
    assert after['var_verdict']['axis_count_parity'] is True
    assert after['var_verdict']['axis_order_parity'] is True
    assert after['var_verdict']['wrong_species_rows'] == 0
    assert after['chunk_decision']['physical_members'] == 1
    assert receipt['writes']['deletions'] == 0
    assert receipt['writes']['collection_writes'] == 0
    assert set(receipt['collections']) == {
        'pert-gym/base-public/20260621',
        'pert-gym/canonical/20260621',
    }
assert plan['mode'] == 'plan' and plan['writes']['obs_revisions'] == 0
assert live['mode'] == 'mutate' and live['writes']['obs_revisions'] in {0, 1}
assert verify['mode'] == 'verify' and verify['writes']['obs_revisions'] == 0
assert verify['replay_noop'] is True


## Rollback and pending work

The predecessor OBS plus the unchanged X and VAR identities remain the rollback path. No payload deletion or Collection mutation occurred. This notebook is complete only when the immutable plan, mutation, and verify receipts above all execute successfully.

In [ ]:
assert live['rollback'] == verify['rollback']
assert live['member_after']['x']['uid'] == 'AVlPOzYBdcrplGXk0000'
assert live['member_after']['var']['uid'] == 'AYnivbGN3JCRzkN70001'
assert verify['member_after']['obs_before']['uid'] == live['member_after']['obs_before']['uid']
print({
    'status': 'PASS',
    'obs_uid': verify['member_after']['obs_before']['uid'],
    'guide_sequence_known_rows': verify['member_after']['source_join']['guide_sequence_known_rows'],
    'x_source_value_mismatches': verify['member_after']['x_source_parity']['source_value_mismatch_count'],
    'replay_noop': verify['replay_noop'],
})
